# 03 - Bronze Ingestion: Kafka streams

Consumes two live Confluent Cloud topics into Bronze:
- `online_orders` — customer online orders
- `inventory_restock` — store restock events

Raw payload is always kept in `raw_payload` even when JSON parsing fails,
so nothing is silently lost — Silver's quality rules are what catch bad
rows later, not this notebook.

In [0]:
# Databricks notebook source
dbutils.widgets.text("catalog", "maven_test")
catalog = dbutils.widgets.get("catalog")

print(f"Running Bronze Kafka ingestion for catalog: {catalog}")

### Connect to Confluent Cloud

Kafka credentials come from the Key Vault-backed secret scope, same as
everything else — never from a local `.env` file.

In [0]:
bootstrap_servers = dbutils.secrets.get(scope="captone_scope", key="kafka-bootstrap-server")
kafka_key = dbutils.secrets.get(scope="captone_scope", key="kafka-api-key")
kafka_secret = dbutils.secrets.get(scope="captone_scope", key="kafka-api-secret")

jaas_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="{kafka_key}" password="{kafka_secret}";'
)

base_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": jaas_config,
    "startingOffsets": "earliest",
    "failOnDataLoss": "false",
}

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.checkpoints")

### Online orders stream

Parses the JSON payload into real columns, de-dupes on `order_id` within
a 10-minute watermark window (a duplicate event shouldn't double-count an
order), and writes both the parsed columns and the raw payload.

In [0]:
from pyspark.sql.functions import col, current_timestamp, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("payment_method", StringType(), True),
    StructField("channel_mode", StringType(), True),
    StructField("event_timestamp", StringType(), True),
])

orders_options = dict(base_options)
orders_options["subscribe"] = "online_orders"

orders_raw = spark.readStream.format("kafka").options(**orders_options).load()

orders_parsed = (
    orders_raw
    .selectExpr(
        "CAST(key AS STRING) as kafka_partition_key",
        "topic as kafka_topic",
        "partition as kafka_partition",
        "offset as kafka_offset",
        "timestamp as kafka_processing_time",
        "CAST(value AS STRING) as raw_payload",
    )
    .withColumn("parsed", from_json(col("raw_payload"), orders_schema))
    .select(
        "kafka_partition_key", "kafka_topic", "kafka_partition",
        "kafka_offset", "kafka_processing_time", "raw_payload", "parsed.*",
    )
    .withColumn("event_timestamp", to_timestamp(col("event_timestamp")))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", col("kafka_topic"))
    .withWatermark("event_timestamp", "10 minutes")
    .dropDuplicatesWithinWatermark(["order_id"])
)

orders_query = (
    orders_parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"/Volumes/{catalog}/bronze/checkpoints/kafka_online_orders/state")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.kafka_online_orders_raw")
)
orders_query.awaitTermination()
print("online_orders rows:", spark.table(f"{catalog}.bronze.kafka_online_orders_raw").count())

### Inventory restock stream

Same pattern, different topic and schema.

In [0]:
inventory_schema = StructType([
    StructField("restock_event_id", StringType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("restock_quantity", IntegerType(), True),
    StructField("restock_type", StringType(), True),
    StructField("event_timestamp", StringType(), True),
])

inventory_options = dict(base_options)
inventory_options["subscribe"] = "inventory_events"

inventory_raw = spark.readStream.format("kafka").options(**inventory_options).load()

inventory_parsed = (
    inventory_raw
    .selectExpr(
        "CAST(key AS STRING) as kafka_partition_key",
        "topic as kafka_topic",
        "partition as kafka_partition",
        "offset as kafka_offset",
        "timestamp as kafka_processing_time",
        "CAST(value AS STRING) as raw_payload",
    )
    .withColumn("parsed", from_json(col("raw_payload"), inventory_schema))
    .select(
        "kafka_partition_key", "kafka_topic", "kafka_partition",
        "kafka_offset", "kafka_processing_time", "raw_payload", "parsed.*",
    )
    .withColumn("event_timestamp", to_timestamp(col("event_timestamp")))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", col("kafka_topic"))
    .withWatermark("event_timestamp", "10 minutes")
    .dropDuplicatesWithinWatermark(["restock_event_id"])
)

inventory_query = (
    inventory_parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"/Volumes/{catalog}/bronze/checkpoints/kafka_inventory_restock/state")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.kafka_inventory_raw")
)
inventory_query.awaitTermination()
print("inventory_restock rows:", spark.table(f"{catalog}.bronze.kafka_inventory_raw").count())

dbutils.notebook.exit("OK")